## IMPORT

In [79]:
%load_ext autoreload
%autoreload

from importlib import reload
import my_sklearn
reload(my_sklearn)

from my_sklearn import (
    my_DecisionTreeClassifier, 
    my_DecisionTreeRegressor, 
    my_ExtraTreeClassifier, 
    my_RandomForestClassifier, 
    my_GradientBoostingClassifier, 
    tune_and_evaluate,
    my_ExtraTreesClassifier
)

import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import CountEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, ExtraTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## DATA

import data from [kaggle](https://www.kaggle.com/competitions/DontGetKicked/data?select=training.csv)

In [80]:
df = pd.read_csv("./data/training.csv")
df.head()

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020


split the data by 'PurchDate' into three DataFrames based on one-third date intervals

In [81]:
df["PurchDate"] = pd.to_datetime(df["PurchDate"]).dt.date

dates = np.sort(df["PurchDate"].unique())
number_of_dates = len(dates)

valid_date = dates[number_of_dates // 3]
test_date = dates[2 * number_of_dates // 3]

train_mask = df["PurchDate"] < valid_date
valid_mask = (df["PurchDate"] >= valid_date) & (df["PurchDate"] < test_date)
test_mask = df["PurchDate"] >= test_date

train_df = df[train_mask]
valid_df = df[valid_mask]
test_df = df[test_mask]

X_train = train_df.drop(columns='IsBadBuy')
y_train = train_df['IsBadBuy']
X_valid = valid_df.drop(columns='IsBadBuy')
y_valid = valid_df['IsBadBuy']
X_test = test_df.drop(columns='IsBadBuy')
y_test = test_df['IsBadBuy']

check the split

In [82]:
print(f"The sum length of splits {len(y_train + y_valid + y_test)} and the sum length of df {len(df)}")
print()
print(f"The max date in train: {X_train["PurchDate"].max()}")
print(f"The min date in valid: {X_valid["PurchDate"].min()}")
print()
print(f"The max date in valid {X_valid["PurchDate"].max()}")
print(f"The min date in test  {X_test["PurchDate"].min()}")

The sum length of splits 72983 and the sum length of df 72983

The max date in train: 2009-09-01
The min date in valid: 2009-09-02

The max date in valid 2010-04-30
The min date in test  2010-05-03


### Preprocess

In [83]:
drop_columns = ['RefId', 'PurchDate', 'VehYear', 'WheelTypeID', 'PRIMEUNIT', 'AUCGUART', 'VNZIP1', 'BYRNO', 'Auction']

X_train = X_train.drop(columns=drop_columns)
X_valid = X_valid.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

In [84]:
unique_cat = X_train.select_dtypes(include='str').nunique()
unique_cat

Make                     31
Model                   795
Trim                    122
SubModel                650
Color                    16
Transmission              2
WheelType                 3
Nationality               4
Size                     12
TopThreeAmericanName      4
VNST                     29
dtype: int64

In [85]:
X_train.select_dtypes(include='number').columns.to_list()

['VehicleAge',
 'VehOdo',
 'MMRAcquisitionAuctionAveragePrice',
 'MMRAcquisitionAuctionCleanPrice',
 'MMRAcquisitionRetailAveragePrice',
 'MMRAcquisitonRetailCleanPrice',
 'MMRCurrentAuctionAveragePrice',
 'MMRCurrentAuctionCleanPrice',
 'MMRCurrentRetailAveragePrice',
 'MMRCurrentRetailCleanPrice',
 'VehBCost',
 'IsOnlineSale',
 'WarrantyCost']

Split the features into categorical and numerical features. Then split the categorical featuresfor into those for OHE and count encoder

In [86]:
categorical_feat_count = [cat for cat in unique_cat.index.to_list() if unique_cat[cat] > 4]
categorical_feat_ohe = [cat for cat in unique_cat.index.to_list() if unique_cat[cat] <= 4]
categorical_feat_ohe.append('IsOnlineSale')

number_feat = X_train.select_dtypes(include='number').columns.to_list()
number_feat.remove("IsOnlineSale")

In [87]:
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe',
            Pipeline([
                ('impute', SimpleImputer(strategy='constant', fill_value='unknown')),
                ('ohe_cat', OneHotEncoder(handle_unknown='ignore'))
            ]), 
            categorical_feat_ohe),
        ('count', CountEncoder(handle_unknown=0, handle_missing='unknown'), categorical_feat_count),
        ('num', SimpleImputer(strategy='median'), number_feat)
    ]
)

In [88]:
preprocessor.fit(X_train)
preprocessor.set_output()

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...), ('count', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature

In [89]:
X_train_processed = pd.DataFrame(
    preprocessor.transform(X_train),
    columns=preprocessor.get_feature_names_out(),
    index=X_train.index)

X_valid_processed = pd.DataFrame(
    preprocessor.transform(X_valid),
    columns=preprocessor.get_feature_names_out(),
    index=X_valid.index)

X_test_processed = pd.DataFrame(
    preprocessor.transform(X_test),
    columns=preprocessor.get_feature_names_out(),
    index=X_test.index)

## My DecisionTree module

In [90]:
def gini(y_actual, y_proba):
    return 2 * roc_auc_score(y_actual, y_proba) - 1

table_gini = pd.DataFrame(
    columns=['train_gini', 'valid_gini']   
)
table_gini.index.name = 'model_name'

def estimate(model_name, model, table_gini, X_train, y_train, X_valid, y_valid):
    model.fit(X_train, y_train)

    proba_train = model.predict_proba(X_train)[:, 1]
    proba_valid = model.predict_proba(X_valid)[:, 1]

    gini_train = gini(y_train, proba_train)
    gini_valid = gini(y_valid, proba_valid)

    new_row = pd.DataFrame(
        [[gini_train, gini_valid]],
        columns=table_gini.columns,
        index=[model_name]
    )

    return pd.concat([table_gini, new_row])

In [91]:
max_d = 4
my_tree = my_DecisionTreeClassifier(max_depth=max_d)
name = f'my_tree_clas_depth_{max_d}'
table_gini = estimate(name, my_tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

tree = DecisionTreeClassifier(max_depth=max_d)
name = f'skl_tree_clas_depth_{max_d}'
table_gini = estimate(name, tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

max_d = 6
my_tree = my_DecisionTreeClassifier(max_depth=max_d)
name = f'my_tree_clas_depth_{max_d}'
table_gini = estimate(name, my_tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

tree = DecisionTreeClassifier(max_depth=max_d)
name = f'skl_tree_clas_depth_{max_d}'
table_gini = estimate(name, tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini


,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461


Both trees get a similar score. This means that the implementation is similar to sklearn's DesicionTree

In [92]:
max_d = 4
my_tree = my_ExtraTreeClassifier(max_depth=max_d)
name = f'my_extra_clas_depth_{max_d}'
table_gini = estimate(name, my_tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

tree = ExtraTreeClassifier(max_depth=max_d)
name = f'skl_extra_clas_depth_{max_d}'
table_gini = estimate(name, tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

max_d = 6
my_tree = my_ExtraTreeClassifier(max_depth=max_d)
name = f'my_extra_clas_depth_{max_d}'
table_gini = estimate(name, my_tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

tree = ExtraTreeClassifier(max_depth=max_d)
name = f'skl_extra_clas_depth_{max_d}'
table_gini = estimate(name, tree, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini


,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461
my_extra_clas_depth_4,0.435751,0.376146
skl_extra_clas_depth_4,0.229091,0.167577
my_extra_clas_depth_6,0.483402,0.421296
skl_extra_clas_depth_6,0.445325,0.42209


My implementation performs better than sklearn's version for both training and validation scores.

### RandomForestClassifier

In [93]:
max_d = 4
my_rft = my_RandomForestClassifier(max_depth=max_d, n_estimators=50, random_state=21)
name = f'my_random_depth_{max_d}'
table_gini = estimate(name, my_rft, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

rft = RandomForestClassifier(max_depth=max_d, n_estimators=50, random_state=21)
name = f'skl_random_depth_{max_d}'
table_gini = estimate(name, rft, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

max_d = 6
my_rft = my_RandomForestClassifier(max_depth=max_d, n_estimators=50, random_state=21)
name = f'my_random_depth_{max_d}'
table_gini = estimate(name, my_rft, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

rft = RandomForestClassifier(max_depth=max_d, n_estimators=50, random_state=21)
name = f'skl_random_depth_{max_d}'
table_gini = estimate(name, rft, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini


,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461
my_extra_clas_depth_4,0.435751,0.376146
skl_extra_clas_depth_4,0.229091,0.167577
my_extra_clas_depth_6,0.483402,0.421296
skl_extra_clas_depth_6,0.445325,0.42209
my_random_depth_4,0.506486,0.458385
skl_random_depth_4,0.509986,0.467191


Both Random Forest models - my implementation and sklearn's - achieve similar scores on the training and validation sets. Both also outperform a single decision tree.

In [94]:
max_d = 4
my_gbdt = my_GradientBoostingClassifier(max_depth=max_d, max_features='sqrt', number_of_trees=50)
name = f"my_gbdt_depth_{max_d}"
table_gini = estimate(name, my_gbdt, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

gbdt = GradientBoostingClassifier(max_depth=max_d, max_features='sqrt', n_estimators=50)
name = f"skl_gbdt_depth_{max_d}"
table_gini = estimate(name, gbdt, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini


,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461
my_extra_clas_depth_4,0.435751,0.376146
skl_extra_clas_depth_4,0.229091,0.167577
my_extra_clas_depth_6,0.483402,0.421296
skl_extra_clas_depth_6,0.445325,0.42209
my_random_depth_4,0.506486,0.458385
skl_random_depth_4,0.509986,0.467191


In [95]:
max_d = 3
my_gbdt = my_GradientBoostingClassifier(max_depth=max_d, max_features='sqrt', number_of_trees=50)
name = f"my_gbdt_depth_{max_d}"
table_gini = estimate(name, my_gbdt, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

gbdt = GradientBoostingClassifier(max_depth=max_d, max_features='sqrt', n_estimators=50)
name = f"skl_gbdt_depth_{max_d}"
table_gini = estimate(name, gbdt, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini

,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461
my_extra_clas_depth_4,0.435751,0.376146
skl_extra_clas_depth_4,0.229091,0.167577
my_extra_clas_depth_6,0.483402,0.421296
skl_extra_clas_depth_6,0.445325,0.42209
my_random_depth_4,0.506486,0.458385
skl_random_depth_4,0.509986,0.467191


Both GBDT models — my implementation and sklearn's — achieve similar scores. Both also outperform a single decision tree.

### ExtraTreesClassifier

In [96]:
max_d = 4
my_extra_trees = my_ExtraTreesClassifier(max_depth=max_d, n_estimators=50, n_times=25, random_state=21)
name = f"my_extrees_depth_{max_d}"
table_gini = estimate(name, my_extra_trees, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

extrees = ExtraTreesClassifier(max_depth=max_d, n_estimators=50, n_jobs=-1, random_state=21)
name = f"skl_extrees_depth_{max_d}"
table_gini = estimate(name, extrees, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

max_d = 6
my_extra_trees = my_ExtraTreesClassifier(max_depth=max_d, n_estimators=50, n_times=25, random_state=21)
name = f"my_extrees_depth_{max_d}"
table_gini = estimate(name, my_extra_trees, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)

extrees = ExtraTreesClassifier(max_depth=max_d, n_estimators=50, n_jobs=-1, random_state=21)
name = f"skl_extrees_depth_{max_d}"
table_gini = estimate(name, extrees, table_gini, X_train_processed, y_train, X_valid_processed, y_valid)
table_gini

,train_gini,valid_gini
my_tree_clas_depth_4,0.471726,0.453562
skl_tree_clas_depth_4,0.471726,0.453082
my_tree_clas_depth_6,0.510499,0.389204
skl_tree_clas_depth_6,0.510499,0.435461
my_extra_clas_depth_4,0.435751,0.376146
skl_extra_clas_depth_4,0.229091,0.167577
my_extra_clas_depth_6,0.483402,0.421296
skl_extra_clas_depth_6,0.445325,0.42209
my_random_depth_4,0.506486,0.458385
skl_random_depth_4,0.509986,0.467191


## LightGBM, Catboost, and XGBoost and RandomForest

In [120]:
results = pd.DataFrame()

In [121]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=21,
    n_jobs=-1
)

xgb_params = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [1, 5, 10], # minimum total sample weight required in a leaf; larger values make the model less flexible.
    "subsample": [0.6, 0.8, 1.0], # fraction of training rows used for each tree.
    "colsample_bytree": [0.6, 0.8, 1.0], #fraction of features used for each tree.
    "gamma": [0, 0.1, 0.5],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 5, 10],
    "n_estimators": [100 ,300, 500, 800]
}

xgb_result = tune_and_evaluate(
    model=xgb_model,
    param_space=xgb_params,
    X_train=X_train_processed,
    y_train=y_train,
    X_valid=X_valid_processed,
    y_valid=y_valid,
    search_type="random",
    n_iter=50,
)

xgb_result = pd.DataFrame([xgb_result])
results = pd.concat([results, xgb_result], ignore_index=True)

In [122]:
lgbm_model = LGBMClassifier(
    objective="binary",
    random_state=21,
    n_jobs=1,
    verbosity=-1
)

lgbm_params = {
    "n_estimators": [100, 300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05],
    "num_leaves": [7, 15, 31], # maximum number of leaves in one tree; lower values strongly reduce overfitting.
    "max_depth": [3, 4, 5, 6],
    "min_child_samples": [20, 50, 100],  # minimum number of observations in a leaf.
    "feature_fraction": [0.6, 0.8, 1.0],
    "bagging_fraction": [0.6, 0.8, 1.0], # fraction of rows sampled for each tree.
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [0, 1, 5, 10]
}

lgbm_result = tune_and_evaluate(
    model=lgbm_model,
    param_space=lgbm_params,
    X_train=X_train_processed,
    y_train=y_train,
    X_valid=X_valid_processed,
    y_valid=y_valid,
    search_type="random",
    n_iter=50,
)

lgbm_result = pd.DataFrame([lgbm_result])
results = pd.concat([results, lgbm_result], ignore_index=True)

In [123]:
cat_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=21,
    verbose=0,
    thread_count=1
)

cat_params = {
    "iterations": [100, 300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05],
    "depth": [4, 5, 6, 7, 8],
    "l2_leaf_reg": [1, 3, 5, 10, 20],
    "random_strength": [0.5, 1, 2, 5],
    "bagging_temperature": [0, 1, 3, 5] # controls Bayesian bootstrap; larger values add more randomness.
}

cat_result = tune_and_evaluate(
    model=cat_model,
    param_space=cat_params,
    X_train=X_train_processed,
    y_train=y_train,
    X_valid=X_valid_processed,
    y_valid=y_valid,
    search_type="random",
    n_iter=50,
)

cat_result = pd.DataFrame([cat_result])
results = pd.concat([results, cat_result], ignore_index=True)

In [124]:
rf_model = RandomForestClassifier(
    random_state=21,
    n_jobs=1
)

rf_params = {
    "n_estimators": [100, 300],
    "max_depth": [3, 4, 5, 6, 7, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", 0.5, 1.0],
    "class_weight": [None, "balanced"]
}

rf_result = tune_and_evaluate(
    model=rf_model,
    param_space=rf_params,
    X_train=X_train_processed,
    y_train=y_train,
    X_valid=X_valid_processed,
    y_valid=y_valid,
    search_type="random",
    n_iter=30,
)

rf_result = pd.DataFrame([rf_result])
results = pd.concat([results, rf_result], ignore_index=True)

In [125]:
results

,model,best_params,best_cv_gini,train_gini,valid_gini,overfit_gap,best_model
0,XGBClassifier,"{'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha...",0.508804,0.632297,0.481288,0.151008,"XGBClassifier(base_score=None, booster=None, c..."
1,LGBMClassifier,"{'subsample': 0.8, 'reg_lambda': 1, 'reg_alpha...",0.506664,0.560017,0.482032,0.077985,"LGBMClassifier(bagging_fraction=0.8, colsample..."
2,CatBoostClassifier,"{'random_strength': 1, 'learning_rate': 0.03, ...",0.509060,0.626239,0.478225,0.148014,"CatBoostClassifier(bagging_temperature=0, dept..."
3,RandomForestClassifier,"{'n_estimators': 300, 'min_samples_split': 10,...",0.500945,0.601271,0.478647,0.122624,"(DecisionTreeClassifier(max_depth=7, max_featu..."


**The best model based on the validation Gini score is LGBMClassifier.**

## Key differences between LightGBM, CatBoost, and XGBoost

| Feature                   | LightGBM                                                                      | CatBoost                                                         | XGBoost                                                             |
| ------------------------- | ----------------------------------------------------------------------------- | ---------------------------------------------------------------- | ------------------------------------------------------------------- |
| Main idea                 | Fast and memory-efficient GBDT implementation                                 | GBDT designed for categorical features                           | Robust and highly optimized GBDT implementation                     |
| Tree growth               | Leaf-wise (best-first) growth                                                 | Symmetric / oblivious trees                                      | Level-wise growth by default                                        |
| Training speed            | Usually the fastest                                                           | Often slower, especially on CPU                                  | Fast, but usually slower than LightGBM                              |
| Categorical features      | Supports categorical features, but they must be encoded as integer categories | Native categorical feature handling                              | Usually requires preprocessing, such as one-hot encoding            |
| Missing values            | Handles missing values automatically                                          | Handles missing values automatically                             | Handles missing values automatically                                |
| Small datasets            | Can overfit because of leaf-wise growth                                       | Often performs very well                                         | Usually stable                                                      |
| Large datasets            | Excellent choice because of high speed and low memory usage                   | Good, but may be slower                                          | Excellent, but can require more resources                           |
| Tree structure            | Asymmetric trees with different depths in different branches                  | Symmetric trees: the same split rule is used at each level       | Usually asymmetric trees                                            |

---

## LightGBM

LightGBM is a gradient boosting library designed for high training speed and efficient memory usage. It is especially useful for large datasets with many numerical features.

Its main feature is **leaf-wise tree growth**. Instead of growing all tree levels evenly, LightGBM chooses the leaf that gives the largest reduction in loss and splits it first.

This can produce more accurate trees, but it can also lead to overfitting on small datasets.

Important LightGBM parameters:

* `n_estimators` — number of boosting trees.
* `learning_rate` — contribution of each new tree to the final model.
* `num_leaves` — maximum number of leaves in one tree. This is one of the most important LightGBM parameters.
* `max_depth` — maximum depth of a tree.
* `min_child_samples` — minimum number of samples required in a leaf.
* `subsample` — fraction of training samples used for each tree.
* `colsample_bytree` — fraction of features used for each tree.
* `reg_alpha` — L1 regularization.
* `reg_lambda` — L2 regularization.

LightGBM can work with categorical features. They should usually be converted to integer category codes first. LightGBM then searches for efficient category splits instead of treating categories as ordinary continuous numbers.

---

## CatBoost categorical features

CatBoost is especially useful when the dataset contains categorical features, such as city, product type, user category, or occupation.

CatBoost does not require manual one-hot encoding for categorical columns. Instead, it uses target-based statistics called **CTR features**.

For example, if the feature is `City`, CatBoost estimates how the target value changes for different cities. However, ordinary target encoding can cause data leakage because the target of the current object may influence its encoded value.

CatBoost avoids this problem with **ordered target statistics**. During training, it uses a random permutation of the dataset. For each object, statistics are calculated only from previous objects in that permutation.

This approach has several advantages:

* It reduces target leakage.
* It works well with high-cardinality categorical features.
* It avoids creating a very large one-hot encoded feature matrix.
* It often gives strong results with minimal preprocessing.

---

## XGBoost DART mode

DART stands for **Dropout Additive Regression Trees**.

In standard gradient boosting, every new tree is added to the predictions of all previous trees:

```text
Prediction = Tree1 + Tree2 + Tree3 + Tree4
```

In DART mode, XGBoost randomly drops some previously built trees while training a new tree:

```text
Prediction = Tree1 + Tree3 + Tree4
```

The new tree is trained using this modified ensemble. This is similar to dropout in neural networks.

Advantages of DART:

* It can reduce overfitting.
* It creates a more diverse ensemble of trees.
* It may improve validation performance on some datasets.

Disadvantages of DART:

* Training is slower than standard `gbtree` boosting.
* Prediction can also be slower.
* It does not always improve the final score.

The main XGBoost booster options are:

* `gbtree` — standard gradient boosted decision trees.
* `dart` — gradient boosting with dropout.
* `gblinear` — linear model booster.

For most tasks, `gbtree` is the default and strongest baseline. DART is worth testing if the standard model overfits.

---

## Which model gives the best result?

There is no universal best model. The best choice depends on the dataset.

* **CatBoost** often performs best when there are many categorical features. Its ordered target statistics and ordered boosting reduce overfitting and allow the model to use categorical variables effectively.
* **LightGBM** often performs best on large datasets with mostly numerical features. It is usually the fastest model, but it may require careful tuning because leaf-wise growth can overfit.
* **XGBoost** is usually a strong and stable baseline. It can achieve excellent results after tuning, but it may be slower than LightGBM and may require categorical feature preprocessing.


## Performance on the test dataset

In [126]:
best_model = results.loc[results['valid_gini'].idxmax(), 'best_model']
train_proba = best_model.predict_proba(X_train_processed)[:, 1]
valid_proba = best_model.predict_proba(X_valid_processed)[:, 1]

X_train_full = pd.concat([X_train_processed, X_valid_processed])
y_train_full = pd.concat([y_train, y_valid])

best_model.fit(X_train_full, y_train_full)

test_proba = best_model.predict_proba(X_test_processed)[:, 1]

print(f"train gini {2 * roc_auc_score(y_train, train_proba) - 1}")
print(f"valid gini {2 * roc_auc_score(y_valid, valid_proba) - 1}")
print(f"test gini {2 * roc_auc_score(y_test, test_proba) - 1}")

train gini 0.5600173703754301
valid gini 0.48203218649541535
test gini 0.5109964836170664


The best model according to the validation Gini score is LGBMClassifier. It achieved the highest validation performance while also having the smallest overfitting gap among the tuned GBDT models, indicating better generalization to unseen data. 

The tuned LGBMClassifier achieved a train Gini of 0.560, a validation Gini of 0.482, and a test Gini of 0.511. The relatively small gap between the training and validation scores indicates that the model does not overfit severely. Moreover, the higher Gini score on the test set demonstrates good generalization performance, making LightGBM the best-performing model for this task.